# 02 — Flash Attention (A-Day 4)

1. Standard attention: materialize N×N matrix and plot O(N²) memory growth.
2. Tiling + online softmax in NumPy (Flash Attention algorithm).
3. vLLM sequence length sweep: VRAM, TTFT, ITL with/without fused kernels.
4. Block size tuning (8, 16, 32).

## 1. Standard attention — memory visualization

Materialize the full N×N attention matrix; plot memory (bytes) vs sequence length.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = 64
seq_lengths = [128, 256, 512, 1024, 2048, 4096]
bytes_per_float = 4  # float32

memory_mb = []
for N in seq_lengths:
    # Attention matrix S = Q @ K^T is (N, N)
    S_size = N * N * bytes_per_float
    memory_mb.append(S_size / (1024**2))

plt.figure(figsize=(8, 4))
plt.plot(seq_lengths, memory_mb, 'o-')
plt.xlabel("Sequence length N")
plt.ylabel("Attention matrix memory (MB)")
plt.title("O(N²) memory: full attention matrix")
plt.grid(True)
plt.show()
print("At N=2048:", memory_mb[seq_lengths.index(2048)], "MB")

In [ ]:
# Exact standard attention for one toy (N=512, d=64)
N, d = 512, 64
np.random.seed(42)
Q = np.random.randn(1, N, d).astype(np.float32) * 0.1
K = np.random.randn(1, N, d).astype(np.float32) * 0.1
V = np.random.randn(1, N, d).astype(np.float32) * 0.1
scale = np.sqrt(d)
S = (Q @ K.transpose(0, 2, 1)) / scale  # (1, N, N)
P = np.exp(S - S.max(axis=-1, keepdims=True))
P = P / P.sum(axis=-1, keepdims=True)
O_standard = P @ V  # (1, N, d)
print("Standard attention output shape:", O_standard.shape)
print("S (attention scores) size:", S.nbytes / 1024, "KB")

## 2. Tiling + online softmax (Flash Attention mental model)

Tile Q, K, V; use running max and running sum for numerical stability; never store full N×N. Verify output matches standard attention.

In [ ]:
def flash_attention_toy(Q, K, V, block_size=64):
    """
    Simplified Flash Attention: tile over K,V, online softmax (running max m, running sum l).
    Q, K, V: (1, N, d)
    """
    assert Q.shape[0] == 1 and K.shape[0] == 1
    N, d = Q.shape[1], Q.shape[2]
    scale = np.sqrt(d)
    O = np.zeros_like(Q)
    m_prev = np.full((1, N), -np.inf, dtype=np.float32)
    l_prev = np.zeros((1, N), dtype=np.float32)

    for start in range(0, N, block_size):
        end = min(start + block_size, N)
        Kb = K[:, start:end, :]   # (1, block, d)
        Vb = V[:, start:end, :]
        S_b = (Q @ Kb.transpose(0, 2, 1)) / scale  # (1, N, block)
        m_curr = np.maximum(m_prev, S_b.max(axis=-1))
        P_b = np.exp(S_b - m_curr[:, :, np.newaxis])
        l_curr = np.exp(m_prev - m_curr) * l_prev + P_b.sum(axis=-1)
        O = O * np.exp(m_prev[:, :, np.newaxis] - m_curr[:, :, np.newaxis]) + P_b @ Vb
        m_prev, l_prev = m_curr, l_curr

    O = O / np.maximum(l_prev[:, :, np.newaxis], 1e-12)
    return O

O_flash = flash_attention_toy(Q[0:1], K[0:1], V[0:1], block_size=64)
diff = np.abs(O_flash - O_standard).max()
print("Max diff vs standard:", diff)
print("Match (within 1e-5):", diff < 1e-5)

## 3. vLLM sequence length sweep

Run inference at 512, 1k, 2k, 4k, 8k; record peak VRAM, TTFT, ITL. Compare with `enforce_eager=True` (no fused Flash Attention) vs default.

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import time

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
seq_lengths = [512, 1024, 2048, 4096, 8192]
base_prompt = "Hello, world. " * 50

In [ ]:
def run_sweep(enforce_eager=False):
    llm = LLM(
        model=MODEL_ID,
        trust_remote_code=True,
        max_model_len=8192,
        enforce_eager=enforce_eager,
    )
    sampling = SamplingParams(temperature=0, max_tokens=10)
    results = []
    for target_len in seq_lengths:
        # Build prompt of ~target_len tokens
        p = (base_prompt * (max(1, target_len // 50)))[:target_len*4]
        tokens = tokenizer.encode(p)
        p = tokenizer.decode(tokens[:target_len])
        t0 = time.perf_counter()
        outs = llm.generate([p], sampling_params=sampling)
        elapsed = time.perf_counter() - t0
        n_out = len(outs[0].outputs[0].token_ids)
        ttft_approx = elapsed / 2  # rough: prefill dominates first token
        itl_approx = (elapsed - ttft_approx) / max(1, n_out - 1) if n_out > 1 else 0
        results.append({"seq_len": len(tokenizer.encode(p)), "time": elapsed, "ttft": ttft_approx, "itl": itl_approx})
    return results

results_default = run_sweep(enforce_eager=False)
print("Default (Flash Attention enabled):")
for r in results_default:
    print(f"  seq={r['seq_len']}: time={r['time']:.2f}s, itl~{r['itl']*1000:.0f}ms")

In [ ]:
results_eager = run_sweep(enforce_eager=True)
print("With enforce_eager=True (no fused Flash Attention):")
for r in results_eager:
    print(f"  seq={r['seq_len']}: time={r['time']:.2f}s, itl~{r['itl']*1000:.0f}ms")

plt.figure(figsize=(8, 4))
lens = [r["seq_len"] for r in results_default]
plt.plot(lens, [r["time"] for r in results_default], 'o-', label="Flash Attention")
plt.plot(lens, [r["time"] for r in results_eager], 's-', label="Eager (no FA)")
plt.xlabel("Prompt length (tokens)")
plt.ylabel("Time (s)")
plt.legend()
plt.title("vLLM: sequence length sweep")
plt.show()

## 4. Block size tuning

Vary `block_size` (in vLLM this is the PagedAttention block size). Measure throughput and note fragmentation.

In [ ]:
block_sizes = [8, 16, 32]
throughputs = []
prompt = "Write a short poem. " * 20
num_tokens = 50

for block_size in block_sizes:
    llm = LLM(
        model=MODEL_ID,
        trust_remote_code=True,
        max_model_len=2048,
        block_size=block_size,
    )
    sampling = SamplingParams(temperature=0, max_tokens=num_tokens)
    t0 = time.perf_counter()
    outs = llm.generate([prompt], sampling_params=sampling)
    elapsed = time.perf_counter() - t0
    n_tok = len(outs[0].outputs[0].token_ids)
    tps = n_tok / elapsed
    throughputs.append(tps)
    print(f"block_size={block_size}: {tps:.1f} tok/s, time={elapsed:.2f}s")

plt.bar([str(b) for b in block_sizes], throughputs)
plt.ylabel("Throughput (tokens/s)")
plt.xlabel("Block size")
plt.title("PagedAttention block size vs throughput")
plt.show()